In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

# 1. นิยามโครงสร้าง Model (ต้องตรงกับตอนที่ Train)
class Numclassification(nn.Module):
    def __init__(self):
        super(Numclassification, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3)
        self.conv3 = nn.Conv2d(32, 64, 3)
        
        self.bn1 = nn.BatchNorm2d(16)
        self.bn2 = nn.BatchNorm2d(32)
        self.bn3 = nn.BatchNorm2d(64)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.3)
        
        # คำนวณมาแล้วว่า Linear input คือ 64*6*6 สำหรับภาพ 64x64
        self.fc1 = nn.Linear(64 * 6 * 6, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        
        x = self.flatten(x)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 2. ตั้งค่า Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. โหลด Model
model = Numclassification()
try:
    # โหลด Weight (ตรวจสอบชื่อไฟล์ .pth ให้ถูกต้อง)
    model.load_state_dict(torch.load("Numclassification.pth", map_location=device))
    model.to(device)
    model.eval()
    print("✅ Model Loaded Successfully!")
except Exception as e:
    print(f"❌ Error Loading Model: {e}")

# 4. เตรียม Transform สำหรับพยากรณ์ (ตัดพวก Random ออก)
predict_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# 5. ฟังก์ชันพยากรณ์
def predict_image(image_path):
    try:
        # เปิดรูปและแปลงเป็น Grayscale ("L")
        img = Image.open(image_path).convert("L")
        img_tensor = predict_transform(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(img_tensor)
            # หาความมั่นใจ (Confidence)
            prob = F.softmax(output, dim=1)
            conf, predicted = torch.max(prob, 1)
            
        class_names = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
        result = predicted.item()
        
        print("-" * 30)
        print(f"📷 Image: {image_path}")
        print(f"🎯 Predicted: {class_names[result]}")
        print(f"📈 Confidence: {conf.item()*100:.2f}%")
        print("-" * 30)
        
    except FileNotFoundError:
        print(f"❌ ไม่พบไฟล์รูปภาพที่: {image_path}")

# --- เรียกใช้งาน ---
# เปลี่ยนชื่อไฟล์รูปภาพของคุณที่นี่
predict_image("img/testNum.png")

✅ Model Loaded Successfully!
------------------------------
📷 Image: img/testNum.png
🎯 Predicted: 7
📈 Confidence: 94.00%
------------------------------
